<div style='background-color:#0d1117; color:white; padding:32px 40px; border-radius:10px; font-family:Arial,sans-serif;'>
<h1 style='color:#58a6ff; margin-bottom:6px; font-size:22px;'>Informe de Práctica — Actividad 4</h1>
<h2 style='color:#ffffff; margin-top:0; font-size:17px; font-weight:normal;'>Infraestructura de Almacenamiento y Procesamiento de Datos IoT</h2>
<hr style='border-color:#30363d; margin:18px 0;'/>
<table style='font-size:14px; color:#8b949e;'>
<tr><td style='padding:3px 24px 3px 0; color:#58a6ff;'><b>Estudiante</b></td><td>Ana María García Arias</td></tr>
<tr><td style='padding:3px 24px 3px 0; color:#58a6ff;'><b>Programa</b></td><td>Maestría en Inteligencia Artificial</td></tr>
<tr><td style='padding:3px 24px 3px 0; color:#58a6ff;'><b>Asignatura</b></td><td>Internet de las Cosas</td></tr>
<tr><td style='padding:3px 24px 3px 0; color:#58a6ff;'><b>Docente</b></td><td>Cristian Duney Bermúdez Quintero</td></tr>
<tr><td style='padding:3px 24px 3px 0; color:#58a6ff;'><b>Fecha</b></td><td>Mayo 2026</td></tr>
</table>
</div>

---
## 1. Introducción

Esta actividad implementa una infraestructura completa de almacenamiento y procesamiento de datos IoT. Un nodo sensor virtual (CounterFit simulando un DHT11) captura temperatura y humedad cada 30 segundos y los envía directamente a **TimescaleDB Cloud**, una base de datos PostgreSQL especializada en series de tiempo. Los datos almacenados se procesan mediante técnicas de preprocesamiento, filtrado y transformación, y se visualizan en un dashboard interactivo construido con **Streamlit**.

```
CounterFit (DHT11 virtual)
        |  nodo_timescale.py — cada 30 s
        v
TimescaleDB Cloud  (hypertable: lecturas_iot)
        |
        +---> Notebook: preprocesamiento, filtrado, transformación
        +---> Dashboard Streamlit  (http://localhost:8501)
```

---
## 2. Componente 1 — Nodo sensor con CounterFit

**CounterFit** es un simulador de hardware IoT que expone sensores virtuales por HTTP. Se configuró un sensor DHT11 virtual con dos canales:

| Campo | Sensor Humedad | Sensor Temperatura |
|---|---|---|
| Pin | 5 | 6 |
| Unidades | Percentage | Celsius |
| Rango simulado | 40 – 90 % | 18 – 35 °C |
| Modo | Random activado | Random activado |

El script `nodo_timescale.py` lee los valores del sensor y los inserta en TimescaleDB con marca de tiempo UTC. Cada inserción exitosa imprime una confirmación en consola.

**Inicio del sistema:**
```bash
# Terminal 1: sensor virtual
.venv\Scripts\counterfit --port 5050

# Terminal 2: captura -> base de datos
.venv\Scripts\python actividad4/nodo_timescale.py --port 5050 --interval-sec 30 --samples 360
```

**Salida en consola (extracto):**
```
Conectando a TimescaleDB Cloud...
Esquema verificado.
[1/360]  2026-05-24T20:01:53Z  T=23.14C  HR=62.78%  -> TimescaleDB OK
[2/360]  2026-05-24T20:02:23Z  T=21.89C  HR=58.41%  -> TimescaleDB OK
...
Total acumulado en lecturas_iot: 360 filas.
```

---
## 3. Componente 2 — TimescaleDB Cloud

TimescaleDB Cloud (Tiger Cloud) fue seleccionado por su diseño nativo para series de tiempo: organiza los datos en una **hypertable** particionada automáticamente por intervalos de tiempo (chunks), lo que permite consultas eficientes sobre ventanas temporales sin escanear toda la tabla.

| Criterio | BD relacional tradicional | TimescaleDB Cloud |
|---|---|---|
| Escalabilidad | Degrada a partir de ~1 M filas | Miles de millones sin degradación |
| Consultas temporales | Escaneo completo | Solo el chunk relevante |
| Funciones time-series | Lógica manual en el cliente | `time_bucket()`, `first()`, `last()` nativos |
| Acceso remoto | Limitado | PostgreSQL estándar (SSL) |

**Esquema implementado:**
```sql
CREATE TABLE lecturas_iot (
    ts              TIMESTAMPTZ NOT NULL,
    temperatura_c   DOUBLE PRECISION NOT NULL,
    humedad_pct     DOUBLE PRECISION NOT NULL,
    fuente          TEXT DEFAULT 'counterfit'
);
SELECT create_hypertable('lecturas_iot', 'ts', if_not_exists => TRUE);
```

### 3.1 Conexión y carga de datos

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join('..', '.env'))

engine = create_engine(
    f"postgresql+psycopg2://{os.environ['TS_USER']}:{os.environ['TS_PASSWORD']}"
    f"@{os.environ['TS_HOST']}:{os.environ.get('TS_PORT','33711')}/{os.environ.get('TS_DB','tsdb')}",
    connect_args={"sslmode": "require"}
)

df = pd.read_sql_query(
    "SELECT ts, temperatura_c, humedad_pct FROM lecturas_iot ORDER BY ts;",
    engine, parse_dates=['ts']
)

print(f"Total de lecturas almacenadas : {len(df)}")
print(f"Periodo registrado            : {df['ts'].min().strftime('%Y-%m-%d %H:%M')} -> {df['ts'].max().strftime('%Y-%m-%d %H:%M')}")
print()
df.tail(5)

---
## 4. Procesamiento de datos

### 4.1 Preprocesamiento

In [ ]:
print("--- Calidad del dataset ---")
print(f"Valores nulos    : {df.isnull().sum().sum()}")
print(f"Filas duplicadas : {df.duplicated().sum()}")

temp_out = df[(df['temperatura_c'] < 17) | (df['temperatura_c'] > 36)]
hum_out  = df[(df['humedad_pct'] < 33)   | (df['humedad_pct'] > 94)]
print(f"Anomalias temp.  : {len(temp_out)} filas")
print(f"Anomalias hum.   : {len(hum_out)} filas")
print()
df[['temperatura_c','humedad_pct']].describe().round(2)

### 4.2 Filtrado con `time_bucket()` de TimescaleDB

In [ ]:
df_hora = pd.read_sql_query("""
    SELECT time_bucket('1 hour', ts) AS hora,
           ROUND(AVG(temperatura_c)::numeric, 2) AS temp_media,
           ROUND(AVG(humedad_pct)::numeric,   2) AS hum_media,
           COUNT(*) AS n_muestras
    FROM lecturas_iot
    GROUP BY hora
    ORDER BY hora;
""", engine, parse_dates=['hora'])

print(f"Cubos horarios generados: {len(df_hora)}")
df_hora

### 4.3 Transformación

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = df.sort_values('ts').reset_index(drop=True)
VENTANA = 5
df['temp_movil'] = df['temperatura_c'].rolling(window=VENTANA, min_periods=1).mean()
df['hum_movil']  = df['humedad_pct'].rolling(window=VENTANA, min_periods=1).mean()

def minmax(s):
    r = s.max() - s.min()
    return (s - s.min()) / r if r > 0 else s * 0

df['temp_norm'] = minmax(df['temperatura_c'])
df['hum_norm']  = minmax(df['humedad_pct'])

# Grafica 1: serie temporal + promedio movil
fig1 = make_subplots(rows=2, cols=1, shared_xaxes=True,
                     subplot_titles=['Temperatura (°C)', 'Humedad relativa (%)'])

fig1.add_trace(go.Scatter(x=df['ts'], y=df['temperatura_c'],
    mode='lines', name='Cruda', line=dict(color='#aaa', width=1, dash='dot'), opacity=0.6), row=1, col=1)
fig1.add_trace(go.Scatter(x=df['ts'], y=df['temp_movil'],
    mode='lines', name=f'Promedio movil (n={VENTANA})', line=dict(color='#e55', width=2)), row=1, col=1)

fig1.add_trace(go.Scatter(x=df['ts'], y=df['humedad_pct'],
    mode='lines', name='Cruda', line=dict(color='#aaa', width=1, dash='dot'), opacity=0.6,
    showlegend=False), row=2, col=1)
fig1.add_trace(go.Scatter(x=df['ts'], y=df['hum_movil'],
    mode='lines', name=f'Promedio movil (n={VENTANA})', line=dict(color='#4af', width=2),
    showlegend=False), row=2, col=1)

fig1.update_layout(title='Serie temporal con promedio movil (ventana = 5 muestras)',
                   height=440, margin=dict(t=60, b=40))
fig1.show()

In [ ]:
# Grafica 2: agregados horarios
fig2 = make_subplots(rows=1, cols=2,
                     subplot_titles=['Temperatura media por hora (°C)',
                                     'Humedad media por hora (%)'])
fig2.add_trace(go.Bar(x=df_hora['hora'], y=df_hora['temp_media'],
    marker_color='#e55', name='Temp'), row=1, col=1)
fig2.add_trace(go.Bar(x=df_hora['hora'], y=df_hora['hum_media'],
    marker_color='#4af', name='Hum'), row=1, col=2)
fig2.update_layout(title='Agregados horarios — time_bucket() de TimescaleDB',
                   height=360, showlegend=False, margin=dict(t=60, b=40))
fig2.show()

In [ ]:
# Grafica 3: normalizacion min-max
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=df['ts'], y=df['temp_norm'],
    mode='lines', name='Temperatura normalizada', line=dict(color='#e55', width=1.8)))
fig3.add_trace(go.Scatter(x=df['ts'], y=df['hum_norm'],
    mode='lines', name='Humedad normalizada', line=dict(color='#4af', width=1.8)))
fig3.update_layout(
    title='Normalizacion min-max — escala [0, 1]',
    xaxis_title='Tiempo', yaxis_title='Valor normalizado',
    height=360, margin=dict(t=60, b=40)
)
fig3.show()

---
## 5. Componente 3 — Dashboard Streamlit

Se implementó un dashboard interactivo (`actividad4/dashboard_iot.py`) que consume directamente la hypertable y presenta los datos organizados en tres pestanas:

| Pestana | Contenido |
|---|---|
| Datos en vivo | Ultima lectura, tabla de las 20 mas recientes, grafica en tiempo real, alertas |
| Analisis exploratorio | Serie temporal completa, histogramas, estadisticas descriptivas |
| Datos procesados | Promedio movil configurable, `time_bucket()` horario, normalizacion min-max |

**Inicio:**
```bash
.venv\Scripts\streamlit run actividad4/dashboard_iot.py
# Abre en http://localhost:8501
```

**Arquitectura del dashboard:**

```
SQLAlchemy engine (cache permanente)
        |
        +-- @st.cache_data(ttl=30s)
               |
               v
        SELECT * FROM lecturas_iot
               |
        +------+------+
        |             |
   Datos en vivo   Analisis / Procesados
   (ultimas 50)    (historico completo)
```

El uso de `st.cache_data` con TTL de 30 segundos evita consultas innecesarias a la base de datos; el refresco automático del dashboard se sincroniza con el intervalo de captura del nodo sensor.

---
## 6. Resultados y conclusiones

### Resumen de resultados

| Etapa | Herramienta | Resultado |
|---|---|---|
| Captura | CounterFit + nodo_timescale.py | 360 lecturas en 3 h, 0 % de error de insercion |
| Almacenamiento | TimescaleDB Cloud — hypertable | Particionado automatico, consultas por chunk |
| Preprocesamiento | pandas | 0 nulos, 0 duplicados, 0 anomalias de rango |
| Filtrado | SQL + `time_bucket()` | Agregados horarios calculados en el servidor |
| Transformacion | rolling mean, min-max | Tendencia suavizada, variables comparables |
| Visualizacion | Streamlit + Plotly | Dashboard interactivo con auto-refresco |

### Conclusiones

1. **TimescaleDB** resuelve de forma nativa las dos limitaciones principales de las bases de datos relacionales para IoT: escalabilidad ante alta frecuencia de escritura y eficiencia en consultas temporales.

2. **`time_bucket()`** reduce el volumen de datos transferidos entre el servidor y el cliente: en lugar de descargar todas las filas para agregar en Python, el motor entrega directamente los promedios por intervalo.

3. **El promedio movil** suaviza las variaciones del sensor sin retrasar la deteccion de tendencias, siendo una tecnica esencial para telemetria IoT con sensores de bajo costo.

4. **La normalizacion min-max** permite comparar temperatura y humedad en la misma escala y es el preprocesamiento estandar para modelos de aprendizaje automatico sobre datos de series de tiempo.

5. **La arquitectura implementada** es escalable a hardware fisico y desplegable en la nube (Streamlit Cloud) sin modificar el pipeline de almacenamiento y procesamiento.

---
*Actividad 4 — Internet de las Cosas | Mayo 2026*